Machine Translation using LSTM encoder-decoder model

Aim: Implement a shallow autoencoder and decoder network for machine translation using the Kaggle English to Hindi Neural Translation dataset. Design and train a model to translate English sentences to Hindi by leveraging the power of autoencoders and decoders

In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [10]:
data = pd.read_csv('English2Hindi.csv', encoding='utf-8').dropna(subset=['English', 'Hindi'])
# Keep short sentence pairs (<= 10 words) and a 10k sample so padding and training stay manageable
eng_len = data['English'].str.split().str.len()
hin_len = data['Hindi'].str.split().str.len()
data = data[(eng_len <= 10) & (hin_len <= 10)].sample(n=10000, random_state=42).reset_index(drop=True)

X_data = data['English'].astype(str).str.lower().str.strip()
y_data = data['Hindi'].astype(str).str.strip().apply(lambda x: 'startseq ' + x + ' endseq')

eng_tokenizer = Tokenizer(filters='')
eng_tokenizer.fit_on_texts(X_data)
num_enc_toks = len(eng_tokenizer.word_index) + 1

hin_tokenizer = Tokenizer(filters='')
hin_tokenizer.fit_on_texts(y_data)
num_dec_toks = len(hin_tokenizer.word_index) + 1

enc_input_sequences = eng_tokenizer.texts_to_sequences(X_data)
dec_sequences = hin_tokenizer.texts_to_sequences(y_data)

max_len_src = max(len(seq) for seq in enc_input_sequences)
max_len_tar = max(len(seq) for seq in dec_sequences)

encoder_input_data = pad_sequences(enc_input_sequences, maxlen=max_len_src, padding='post')
padded_dec_sequences = pad_sequences(dec_sequences, maxlen=max_len_tar, padding='post')

decoder_input_data = padded_dec_sequences[:, :-1]
decoder_output_data = padded_dec_sequences[:, 1:]

In [11]:
max_len_tar = max_len_tar - 1

latent_dim = 250

enc_inps = Input(shape=(max_len_src,))
enc_emb = Embedding(num_enc_toks, latent_dim, mask_zero=True)(enc_inps)
enc_outputs, st_h, st_c = LSTM(latent_dim, return_state=True)(enc_emb)
enc_states = [st_h, st_c]

dec_inps = Input(shape=(max_len_tar,))
dec_emb = Embedding(num_dec_toks, latent_dim, mask_zero=True)(dec_inps)
dec_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
dec_outputs, _, _ = dec_lstm(dec_emb, initial_state=enc_states)
dec_dense = Dense(num_dec_toks, activation='softmax')
dec_outputs = dec_dense(dec_outputs)

model = Model([enc_inps, dec_inps], dec_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [12]:
print("Training model...")
model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_output_data,
    batch_size=16,
    epochs=50,
    validation_split=0.2
)

Training model...
Epoch 1/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 68s 131ms/step - accuracy: 0.2473 - loss: 6.1231 - val_accuracy: 0.2653 - val_loss: 5.7679
Epoch 2/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 63s 125ms/step - accuracy: 0.2703 - loss: 5.2699 - val_accuracy: 0.2780 - val_loss: 5.6280
Epoch 3/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 72s 144ms/step - accuracy: 0.2919 - loss: 4.8209 - val_accuracy: 0.2887 - val_loss: 5.6041
Epoch 4/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 72s 145ms/step - accuracy: 0.3181 - loss: 4.3978 - val_accuracy: 0.3027 - val_loss: 5.5901
Epoch 5/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 71s 143ms/step - accuracy: 0.3469 - loss: 3.9633 - val_accuracy: 0.3068 - val_loss: 5.6542
Epoch 6/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 73s 145ms/step - accuracy: 0.3809 - loss: 3.5291 - val_accuracy: 0.3130 - val_loss: 5.7261
Epoch 7/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 67s 135ms/step - accuracy: 0.4242 - loss: 3.1044 - val_accuracy: 0.3226 - val_loss: 5.7929
Epoch 8/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 71s 143ms/step - accuracy

In [15]:
enc_model = Model(enc_inps, enc_states)

dec_st_inp_h = Input(shape=(latent_dim,))
dec_st_inp_c = Input(shape=(latent_dim,))
dec_states_inps = [dec_st_inp_h, dec_st_inp_c]

dec_emb2 = Embedding(num_dec_toks, latent_dim, mask_zero=True)(dec_inps)

dec_emb2 = model.layers[3](dec_inps)

dec_outputs2, st_h2, st_c2 = model.layers[5](dec_emb2, initial_state=dec_states_inps)
dec_outputs2 = model.layers[6](dec_outputs2)

dec_model = Model([dec_inps] + dec_states_inps, [dec_outputs2] + [st_h2, st_c2])

rev_tar_char_idx = {i: word for word, i in hin_tokenizer.word_index.items()}

def translate(input_text):
    seq = eng_tokenizer.texts_to_sequences([input_text])
    padded_seq = pad_sequences(seq, maxlen=max_len_src, padding='post')

    states_value = enc_model.predict(padded_seq, verbose=0)
    tar_seq = np.zeros((1, max_len_tar))
    tar_seq[0, 0] = hin_tokenizer.word_index['startseq']

    dec_sen = []

    for t in range(max_len_tar - 1):
        output_toks, h, c = dec_model.predict([tar_seq] + states_value, verbose=0)
        sampled_tok_idx = np.argmax(output_toks[0, t, :])

        if sampled_tok_idx == 0:
            break

        sampled_char = rev_tar_char_idx.get(sampled_tok_idx, '')

        if sampled_char == 'endseq':
            break

        dec_sen.append(sampled_char)    
        tar_seq[0, t + 1] = sampled_tok_idx

    return ' '.join(dec_sen)

print("\n--- Translation Test ---")
for i in range(0,5):
  test_phrase = X_data.iloc[i]
  print(f"English Input: {test_phrase}")
  print(f"Predicted Hindi: {translate(test_phrase)}")
  print(f"Actual Expected Hindi: {data['Hindi'].iloc[i]}")


--- Translation Test ---
English Input: you know, what i do is write for children,
Predicted Hindi: मैं बच्चो के लिए कहानियाँ लिखता हूँ ,
Actual Expected Hindi: मैं बच्चो के लिए कहानियाँ लिखता हूँ ,
English Input: is to get them a blueprint
Predicted Hindi: ब्लूप्रिंट को लाना है
Actual Expected Hindi: ब्लूप्रिंट को लाना है
English Input: in terms of your youthfulness, both physical and mental,
Predicted Hindi: शारीरिक और मानसिक तौर से जवान,
Actual Expected Hindi: शारीरिक और मानसिक तौर से जवान,
English Input: going into protests as a couple of hundred,
Predicted Hindi: वे लोग विरोध प्रदर्शन करने जाते थे ,
Actual Expected Hindi: वे लोग विरोध प्रदर्शन करने जाते थे ,
English Input: (laughter)
Predicted Hindi: (हंसी)
Actual Expected Hindi: (हँसी)
